# Field Service Toy Simulation

This notebook demonstrates a small illustrative field-service model.

**Important:** this is not a reproduction of the paper, uses no paper data, and makes no empirical claim about the paper's findings. The parameters are synthetic and are used only to make the main trade-off easier to inspect.

## Model Idea

The toy model compares different `dedicated_technician_ratio` values under the same synthetic scenario. Flexible technicians can handle both emergency and PM jobs; PM-focused technicians handle planned maintenance. Delayed PM can increase avoidable emergency pressure.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from field_service_toy_simulation import SimulationConfig, run_ratio_sweep

base_config = SimulationConfig(
    days=90,
    workload=2.1,
    preventive_maintenance_interval=30,
    machine_reliability_proxy=0.55,
    seed=42,
)

ratios = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
summaries = run_ratio_sweep(base_config, ratios)
summaries[:2]

## Summary Table

Lower is better for `emergency_response_proxy` and `avg_penalty_like_score`. Higher is better for `avg_pm_timeliness_proxy`.

In [ ]:
headers = [
    "dedicated_ratio",
    "emergency_response_proxy",
    "pm_timeliness_proxy",
    "penalty_like_score",
    "utilization",
    "final_pm_backlog",
    "final_emergency_backlog",
]

print(" | ".join(headers))
print(" | ".join(["---"] * len(headers)))
for row in summaries:
    print(
        " | ".join(
            [
                f"{row.dedicated_technician_ratio:.2f}",
                f"{row.avg_emergency_response_proxy:.2f}",
                f"{row.avg_pm_timeliness_proxy:.2f}",
                f"{row.avg_penalty_like_score:.2f}",
                f"{row.avg_utilization:.2f}",
                str(row.final_pm_backlog),
                str(row.final_emergency_backlog),
            ]
        )
    )

## Visual Checks

These plots are meant to make the illustrative mechanism visible. They should not be read as empirical findings.

In [ ]:
import matplotlib.pyplot as plt

x = [row.dedicated_technician_ratio for row in summaries]
emergency = [row.avg_emergency_response_proxy for row in summaries]
pm_timeliness = [row.avg_pm_timeliness_proxy for row in summaries]
penalty = [row.avg_penalty_like_score for row in summaries]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(x, emergency, marker="o", color="#2563eb")
axes[0].set_title("Emergency response proxy")
axes[0].set_xlabel("Dedicated technician ratio")
axes[0].set_ylabel("Lower is better")

axes[1].plot(x, pm_timeliness, marker="o", color="#16a34a")
axes[1].set_title("PM timeliness proxy")
axes[1].set_xlabel("Dedicated technician ratio")
axes[1].set_ylabel("Higher is better")
axes[1].set_ylim(0, 1.05)

axes[2].plot(x, penalty, marker="o", color="#dc2626")
axes[2].set_title("Total penalty-like score")
axes[2].set_xlabel("Dedicated technician ratio")
axes[2].set_ylabel("Lower is better")

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.suptitle("Illustrative toy model - not a reproduction of the paper", y=1.05)
plt.tight_layout()
plt.show()

## Parameters to Try

- Increase `workload` to stress both PM and emergency capacity.
- Increase `preventive_maintenance_interval` to reduce scheduled PM demand but increase the consequences of late PM in some settings.
- Increase `machine_reliability_proxy` to reduce base emergency pressure.
- Change `dedicated_technician_ratio` to inspect the PM versus emergency response trade-off.